# Veritas executor: end-to-end test

Automated evaluation of lab genome submissions (PathoEQA + Veritas)

## What are we building?

- PathoEQA receives genome submissions from laboratories
- Veritas scores each submission against a known-truth reference
- The **executor** is the piece in between: it takes a work order, runs Veritas on a cloud machine, and reports back
- Goal: no heavy analysis on the web server, and every step traceable

## What this test shows

- A real, public sample (SARS-CoV-2, Gamma variant); the lab submission is **simulated** with known errors
- The PathoEQA server is replaced by a stand-in, so the executor can be tested on its own
- The executor: receives the work order, downloads and verifies every input, runs Veritas, reports progress step by step
- A corrupted input is caught **before** any analysis runs

## Live: start the stand-in server

In [1]:
import subprocess, sys, os, time
MOCK = os.path.expanduser("~/Workspace/veritas/pathoeqa-mock")
server = subprocess.Popen([sys.executable, "mock_pathoeqa_server.py"], cwd=MOCK,
                          stdout=open("/tmp/mock.log", "w"), stderr=subprocess.STDOUT)
time.sleep(1.5)
print(open("/tmp/mock.log").read())

Mock PathoEQA on https://localhost:8443
  good attempt: 11111111-1111-4111-8111-111111111111
  bad  attempt: 22222222-2222-4222-8222-222222222222



## Live: run one evaluation

In [2]:
%%bash
cd ~/Workspace/veritas/pathoeqa-mock && ./run_demo.sh good

2026-11-03 10:03:18,306 INFO veritas_runner: Using OIDC token supplied by the caller.
2026-11-03 10:03:18,306 INFO veritas_runner.pathoeqa: Fetching manifest for attempt_id=11111111-1111-4111-8111-111111111111
2026-11-03 10:03:18,315 WARNING veritas_runner.retry: manifest fetch failed transiently (upstream_unavailable); retry 1/2 in 0.5s
2026-11-03 10:03:18,815 INFO veritas_runner.pathoeqa: Fetching manifest for attempt_id=11111111-1111-4111-8111-111111111111
2026-11-03 10:03:18,820 WARNING veritas_runner.retry: manifest fetch failed transiently (upstream_unavailable); retry 2/2 in 1.0s
2026-11-03 10:03:19,820 INFO veritas_runner.pathoeqa: Fetching manifest for attempt_id=11111111-1111-4111-8111-111111111111
2026-11-03 10:03:19,826 ERROR veritas_runner: Could not reach PathoEQA for manifest fetch. [class=upstream_unavailable, attempt_id=11111111-1111-4111-8111-111111111111]


exit code: 30   result: /tmp/veritas-demo-good.json
{"attempt_id":"11111111-1111-4111-8111-111111111111","terminal_state":"attempt_failed","failure_class":"upstream_unavailable","spec_failure_class":"executor_unavailable","duration_ms":null,"message":"Could not reach PathoEQA for manifest fetch. [class=upstream_unavailable, attempt_id=11111111-1111-4111-8111-111111111111]","dry_run":false}


In [3]:
%%bash
# What the server received, in order
grep CALLBACK /tmp/mock.log | tac | awk '{print} /attempt_started/{exit}' | tac

In [4]:
%%bash
# Variant-calling metrics produced by Veritas
column -t -s$'\t' /tmp/veritas-demo/11111111-1111-4111-8111-111111111111/out/SEARCH-8113/output/metrics.tsv

column: /tmp/veritas-demo/11111111-1111-4111-8111-111111111111/out/SEARCH-8113/output/metrics.tsv: No such file or directory


CalledProcessError: Command 'b"# Variant-calling metrics produced by Veritas\ncolumn -t -s$'\\t' /tmp/veritas-demo/11111111-1111-4111-8111-111111111111/out/SEARCH-8113/output/metrics.tsv\n"' returned non-zero exit status 1.

## Live: a corrupted input (optional)

`checksum_mismatch`: replaced the truth VCF's real SHA-256 with 64 zeros in the manifest, simulating a file that was corrupted in transit or swapped for another one

In [ ]:
%%bash
cd ~/Workspace/veritas/pathoeqa-mock && ./run_demo.sh bad 2>&1 | tail -4
echo; grep CALLBACK /tmp/mock.log | tail -3

2026-10-06 12:12:31,730 INFO veritas_runner.pathoeqa: Callback event_type=sample_failed attempt_id=22222222-2222-4222-8222-222222222222 sample_run_id=SEARCH-8113 event_id=b05baf82-74ba-44d6-a720-aabf1c488ec1
2026-10-06 12:12:32,276 INFO veritas_runner.pathoeqa: Callback event_type=attempt_failed attempt_id=22222222-2222-4222-8222-222222222222 sample_run_id=None event_id=bedc166d-671b-4e7e-b2fd-43ba1b76c34f
exit code: 1   result: /tmp/veritas-demo-bad.json
{"attempt_id":"22222222-2222-4222-8222-222222222222","terminal_state":"attempt_failed","duration_ms":36,"veritas_version":"unknown","samples_total":1,"samples_completed":0,"samples":[{"sample_run_id":"SEARCH-8113","terminal_state":"sample_failed","failure_class":"checksum_mismatch","duration_ms":18}],"dry_run":false}

<<< CALLBACK #6 sample_started  sample=SEARCH-8113  idem=0d72d7e2-3941-4fec-bc24-2c14dba5f733
<<< CALLBACK #7 sample_failed  sample=SEARCH-8113  idem=b05baf82-74ba-44d6-a720-aabf1c488ec1
<<< CALLBACK #8 attempt_failed  s

## What we gained

- The executor works end to end **before** the real server exists: less integration risk later
- Every input is checked against a fingerprint; bad data stops the run safely
- Progress is reported per step, so partial work is not lost
- Clear, traceable results per sample

## Next steps

1. Bring the PathoEQA server online (deployment in progress)
2. Adopt the finalized data contract between PathoEQA and Veritas (SPEC-16)
3. Harden the code: review, automated tests, cleaner error reporting
4. Run the full chain on GitHub Actions together with PathoEQA
5. Send results back to PathoEQA and show them to administrators
6. Use real laboratory data only after the formal confidentiality decision

In [ ]:
server.terminate()  # stop the stand-in server